# 03-area: stage 1 on the full area

Stage 1 runs the chain on the whole area with the code in `src/catchment/`. This first section is the **snapping fix**
(#14): the before and after, on the slice and on the area. The area demand map and the full-area solve time (#15) follow
it; the spot-check comes in another issue.

Stage 0.5 found about one short pair in seven riding more than 300 m out and back at one end, from snapping onto the far
carriageway of a divided road. `catchment-build reach` now snaps each candidate site and each hex centre to two roads
and keeps the fastest of the four times for a pair; the 17-min cap applies after the minimum. This notebook reads what
`catchment-build` wrote (the after), rebuilds the one-snap table in the same session (the before), and compares them.

In [ ]:
import os
import resource
import tempfile
import time
from collections import defaultdict
from dataclasses import dataclass, replace
from pathlib import Path

import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import ortools
import pandas as pd
from ortools.sat.python import cp_model
from pyproj import Transformer
from shapely.geometry import Polygon, box

from catchment import assumptions
from catchment.area import load_area
from catchment.fetch import DATA
from catchment.assumptions import (BRAND_COUNTS, CAPACITY, MULTIPLIER, N_AREA, ORDERS_PER_STORE_DAY, PICK_AND_PACK,
                                   PROMISE, SCALE, SEED, SOLVER_SEED, TIME_LIMIT, WORKERS)
from catchment.reach import build_reach, n_top, served_osrm

FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
SOLVE_BAR_S = 10   # a plan for the default scenario must return within this many seconds
MAX_MIN = assumptions.MAX_PAIR_MIN
DEFAULT_MIN = (assumptions.PROMISE - assumptions.PICK_AND_PACK) / assumptions.MULTIPLIER   # 5.0 free-flow minutes
area = load_area()
slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)   # the 0.5 slice
demand = pd.read_parquet(DATA / "demand-r9.parquet")
cand_cmd = pd.read_parquet(DATA / "candidates-r9-x30.parquet")        # catchment-build's outputs (the after)
table_cmd = pd.read_parquet(DATA / "table-r9-x30.parquet")
pts = gpd.read_parquet(DATA / "overture_places.parquet")
places = pd.DataFrame({"lon": pts.geometry.x, "lat": pts.geometry.y, "operating_status": pts.operating_status})   # as build.py loads them
to_utm = Transformer.from_crs(4326, 32643, always_xy=True)   # metres, for distances along the route

## Assumptions

One row of `assumptions.table()` is new in this section. **Assumption** means a labelled choice with no external
source.

In [ ]:
rows = assumptions.table()
pd.set_option("display.max_colwidth", None)
display(rows[rows.iloc[:, 0] == "Second road for snapping"].style.hide(axis="index").set_properties(**{"text-align": "left"}))

## Before and after, measured in one session

Inside one `served_osrm` block the table is built twice with the same function: `snaps=1` (the nearest road only, the
#13 build) and `snaps=2` (the fix). The two-snap table must equal the one `catchment-build` wrote; that check is what
lets the rest of the notebook read the command's output. The block also snaps every candidate site and hex centre to two
roads (to count how many got a distinct second road) and runs the out-and-back sample from the slice section below.

In [ ]:
def route_pair(url, src, dst):
    """One /route between two snapped points: minutes, path m while within 100 m of the start, path m after first
    reaching 100 m of the end (the 01-reach method). > 300 m at an end counts as out-and-back."""
    import requests
    rt = requests.get(f"{url}/route/v1/driving/{src[0]:.6f},{src[1]:.6f};{dst[0]:.6f},{dst[1]:.6f}",
                      params={"overview": "full", "geometries": "geojson"}, timeout=300).json()
    assert rt["code"] == "Ok", rt
    (ax, ay), (bx, by) = (to_utm.transform(*w["location"]) for w in rt["waypoints"])
    x, y = to_utm.transform(*np.array(rt["routes"][0]["geometry"]["coordinates"]).T)
    along = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    leave = along[np.where(np.hypot(x - ax, y - ay) <= 100)[0].max()]
    arrive = along[-1] - along[np.where(np.hypot(x - bx, y - by) <= 100)[0].min()]
    return rt["routes"][0]["duration"] / 60, leave, arrive

def in_poly(df, poly):
    return gpd.GeoSeries(gpd.points_from_xy(df.lon, df.lat), index=df.index).within(poly)

def timed(snaps, router):
    t0 = time.perf_counter()
    c, t = build_reach(area, demand, places, 9, 30, router, snaps=snaps)
    return c, t, time.perf_counter() - t0

area_hexes = demand[in_poly(demand, area)]
slice_hexes = area_hexes[in_poly(area_hexes, slice_poly)]
slice_cands = cand_cmd[in_poly(cand_cmd, slice_poly)]

with served_osrm(DATA) as router:
    cand_before, table_before, secs_before = timed(1, router)
    cand_after, table_after, secs_after = timed(2, router)
    c_lon, c_lat, _ = router.snap(cand_cmd.lon.to_numpy(), cand_cmd.lat.to_numpy(), k=2)
    h_lon, h_lat, _ = router.snap(area_hexes.lon.to_numpy(), area_hexes.lat.to_numpy(), k=2)
    # out-and-back sample: 200 slice pairs with BEFORE minutes <= 6, candidate != hex, seed 42
    in_slice = table_before.candidate.isin(slice_cands.index) & table_before.hex.isin(slice_hexes.index)
    short = table_before[in_slice & (table_before.minutes <= 6) & (table_before.candidate != table_before.hex)]
    c_pos = {h: i for i, h in enumerate(cand_cmd.index)}
    h_pos = {h: i for i, h in enumerate(area_hexes.index)}
    samples = []
    for _, p in short.sample(200, random_state=42).iterrows():
        i, j = c_pos[p.candidate], h_pos[p.hex]
        c_pts = [(c_lon[i, a], c_lat[i, a]) for a in range(2)]
        h_pts = [(h_lon[j, b], h_lat[j, b]) for b in range(2)]
        before = route_pair(router.url, c_pts[0], h_pts[0])
        after = min((route_pair(router.url, c, h) for c in c_pts for h in h_pts), key=lambda r: r[0])
        samples.append({"candidate": p.candidate, "hex": p.hex, "table min before": p.minutes,
                        "before min": before[0], "before leave m": before[1], "before arrive m": before[2],
                        "after min": after[0], "after leave m": after[1], "after arrive m": after[2]})
peak_bytes = router.peak_bytes   # osrm-routed's peak over both builds, the snaps and the sample; read when the block stopped
samples = pd.DataFrame(samples)

# the notebook must read what the command builds
assert list(cand_after.index) == list(cand_cmd.index)
key = ["candidate", "hex"]
a, b = table_after.sort_values(key).reset_index(drop=True), table_cmd.sort_values(key).reset_index(drop=True)
assert len(a) == len(b) and (a[key] == b[key]).all().all() and np.allclose(a.minutes, b.minutes, atol=1e-4), "snaps=2 differs from catchment-build"
print("snaps=2 table equals catchment-build's:", len(a), "pairs")

### Full area

Pairs, table size on disk, build time and memory, before and after. The before table is written to a temporary file to
measure its size; the after size is the file `catchment-build` wrote. The times are `build_reach` alone (OSRM already
serving), so a little under `catchment-build`'s wall time.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    before_path = Path(tmp) / "table-before.parquet"
    table_before.to_parquet(before_path)
    size_before = before_path.stat().st_size
size_after = (DATA / "table-r9-x30.parquet").stat().st_size
full = pd.DataFrame({
    "before (1 snap)": [len(table_before), (table_before.minutes <= DEFAULT_MIN).sum(), size_before / 1e6, secs_before],
    "after (2 snaps)": [len(table_after), (table_after.minutes <= DEFAULT_MIN).sum(), size_after / 1e6, secs_after],
}, index=[f"pairs within {MAX_MIN} min", f"pairs within {DEFAULT_MIN:.1f} min (default scenario)", "table on disk, MB", "build_reach time, s"])
full["change"] = full["after (2 snaps)"] / full["before (1 snap)"] - 1
display(full.round(2))
print(f"osrm-routed peak (both builds, snaps and sample): {peak_bytes / 1e6:,.0f} MB")
print(f"this notebook process's peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:,.0f} MB (holds both tables and the places)")

# second road: how many points got a snap point different from their nearest road's
def distinct(lon, lat):
    return ((lon[:, 0] != lon[:, 1]) | (lat[:, 0] != lat[:, 1]))
d_c, d_h = distinct(c_lon, c_lat), distinct(h_lon, h_lat)
print(f"distinct second road: {d_c.sum():,} of {len(d_c):,} candidate sites ({d_c.mean():.1%}), "
      f"{d_h.sum():,} of {len(d_h):,} hex centres ({d_h.mean():.1%})")

### Slice

The same slice as 0.5. Pairs whose candidate site and hex both have centres in the slice, outer-joined before and after
on (candidate, hex). A pair moved if its time changed by more than 0.01 min; the histogram is over moved pairs that were
already within 17 min before.

In [ ]:
def slice_only(t):
    return t[t.candidate.isin(slice_cands.index) & t.hex.isin(slice_hexes.index)]

j = slice_only(table_before).merge(slice_only(table_after), on=key, how="outer", suffixes=("_b", "_a"))
both = j.dropna(subset=["minutes_b", "minutes_a"])
saved = both.minutes_b - both.minutes_a
moved = saved[saved.abs() > 0.01]
faster, slower = moved[moved > 0], moved[moved < 0]
new_17 = j.minutes_b.isna().sum()
new_5 = ((j.minutes_a <= DEFAULT_MIN) & ~(j.minutes_b <= DEFAULT_MIN)).sum()
print(f"slice pairs: {len(slice_only(table_before)):,} before, {len(slice_only(table_after)):,} after; only before: {j.minutes_a.isna().sum()}")
print(f"moved (|d| > 0.01 min): {len(moved):,} of {len(both):,} ({len(moved) / len(both):.1%}); slower: {len(slower)} (worst {slower.min() if len(slower) else 0:.3f} min)")
print(f"minutes saved over faster pairs: median {faster.median():.2f}, 90th pct {faster.quantile(.9):.2f}, max {faster.max():.2f}")
print(f"faster by more than 0.5 min: {(moved > 0.5).sum():,} ({(moved > 0.5).sum() / len(both):.1%}); by more than 1 min: {(moved > 1).sum():,} ({(moved > 1).sum() / len(both):.1%})")
print(f"newly within {MAX_MIN} min: {new_17:,}; newly within {DEFAULT_MIN:.1f} min: {new_5:,}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(faster, bins=40, color="#3b6ea5")
ax.set_xlabel("minutes saved by the fastest of four (before - after)")
ax.set_ylabel("pairs")
ax.set_title(f"Slice: {len(faster):,} pairs got faster ({len(faster) / len(both):.1%} of {len(both):,})")
fig.tight_layout()
fig.savefig(FIGS / "03-snapping.png", dpi=150)
plt.show()

![Minutes saved on the slice's moved pairs](figures/03-snapping.png)

### Out-and-back on short pairs

200 slice pairs (seed 42) with free-flow time at most 6 min before and candidate site different from hex. Before: one
`/route` from the candidate's nearest snap to the hex's nearest snap. After: the fastest of the four `/route` calls
between the two snaps of each end. Path while still within 100 m of the start, and after first reaching 100 m of the
end, as in 01-reach; more than 300 m counts as out-and-back.

In [ ]:
for s in ("before", "after"):
    samples[f"{s} path m"] = samples[[f"{s} leave m", f"{s} arrive m"]].max(axis=1)
rows = {}
for s in ("before", "after"):
    rows[s] = {"pairs": len(samples),
               "> 300 m at either end": int((samples[f"{s} path m"] > 300).sum()),
               "start end (candidate)": int((samples[f"{s} leave m"] > 300).sum()),
               "hex end": int((samples[f"{s} arrive m"] > 300).sum()),
               "median start/end path m": samples[f"{s} path m"].median()}
display(pd.DataFrame(rows).round(0))
print("share > 300 m: before", f"{(samples['before path m'] > 300).mean():.1%}", "after", f"{(samples['after path m'] > 300).mean():.1%}")
print(f"max |before route min - table min|: {(samples['before min'] - samples['table min before']).abs().max():.3f}; "
      f"pairs where after is slower than before: {int((samples['after min'] > samples['before min'] + 0.01).sum())}")
print(f"still > 300 m after: {int((samples['after path m'] > 300).sum())}; of them, fixed in minutes (saved > 0.01): "
      f"{int(((samples['after path m'] > 300) & (samples['before min'] - samples['after min'] > 0.01)).sum())}")

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not this one). The two-snap table built here equals `catchment-build`'s (1,187,469 pairs, same minutes).

**Full area** (res 9, X 30, 1,458 candidate sites, 4,859 hexes):

| | before (1 snap) | after (2 snaps) | change |
|---|---:|---:|---:|
| pairs within 17 min | 1,137,695 | 1,187,469 | +4.4% |
| pairs within 5.0 min (default scenario) | 50,162 | 59,855 | +19.3% |
| table on disk | 3.69 MB | 3.82 MB | +3.5% |
| `build_reach` time (OSRM already serving) | 59 s | 159 s | 2.7 times |
| `osrm-routed` peak memory (both builds, snaps and sample) | | 410 MB | |

The notebook process peaked at 446 MB (it holds both tables and the places); `catchment-build reach` itself peaked at
505 MB and took 179 s wall time, OSRM serve included. The build fits 8 GB with a wide margin: more roads per point
change the number of `/table` requests (four times as many), not the memory per request. The time varies with machine
load: a first run of this notebook measured 70 s and 192 s. The one-snap table here has 1,251 more pairs than the #13
build (1,136,444); #13 snapped candidate sites only and let OSRM snap the hex centres inside `/table`, while
`snaps=1` here snaps both ends the same way as `snaps=2`, so it is the like-for-like before.

**Second road.** 1,410 of 1,458 candidate sites (96.7%) and 3,264 of 4,859 hex centres (67.2%) got a snap point
different from the nearest road's. The rest repeat the nearest road: no second road within 100 m lying more than 5 m
across it (open land at the area's edge, a lane with nothing parallel). Probably sites sit in the built-up core, where roads run parallel, and hex centres include the sparse outskirts (not checked).

**Slice** (same slice as 0.5: 299,666 pairs before, 308,100 after; nothing in the before table dropped out):

| | |
|---|---:|
| pairs that moved (more than 0.01 min) | 225,317 of 299,666 (75.2%) |
| faster by more than 0.5 min / more than 1 min | 64,013 (21.4%) / 27,985 (9.3%) |
| minutes saved over moved pairs: median / 90th percentile / max | 0.25 / 1.18 / 11.16 |
| pairs slower | 0 |
| newly within 17 min | 8,434 |
| newly within 5.0 min | 3,711 |

Most pairs moved, but by little: the median saving is 15 s, because the fastest of four takes a few seconds off
almost any pair by choosing a better entry onto the road, not only by avoiding a far carriageway. The tail is the fix:
a tenth of the pairs gain over a minute and the largest gain is 11 min, and 3,711 pairs enter the default scenario's
5.0-minute catchments (1.2% of the slice's pairs; over the whole area the default-scenario pairs grow by 9,693). Nothing got slower, as it must when
the minimum includes the old time.

**Out-and-back on 200 short slice pairs** (before time at most 6 min, seed 42; "before" route time equals the table's
exactly, max difference 0.000 min):

| | before | after |
|---|---:|---:|
| more than 300 m at either end | 29 (14.5%) | 19 (9.5%) |
| start end (candidate site) | 14 | 6 |
| hex end | 16 | 13 |
| median start/end path | 114 m | 108 m |

(An end count adds to more than the pair count because a pair can loop at both ends.) The share drops from 14.5% to
9.5%, mainly at the candidate end (14 to 6), where the two snaps were chosen to give a divided road's two carriageways;
0.5 measured 15% (6 of 40) on a smaller sample. It is not gone. **19 pairs still ride more than 300 m at an end, and 16
of them are faster than before**: taking a better road did not remove the detour. Likely reasons, not checked here: a
one-way or a median with no opening, where both carriageways of a divided road lead the same way out and back; a second
road that is none (the nearest road repeats) or is a service road; and the hex end, where 3,264 of 4,859 centres have a
distinct second road and the rest, 1,595 hexes, only the nearest one. Whether 9.5% matters for who is reachable is for the spot-check; the
snapping row of the assumptions table already says it is a fix by geometry, not by the road's name.

## Area demand map

Demand per res-9 hex across the whole area, in orders a day, at the default capture share (read from the demand
table's attributes). Every hex of `demand-r9` is drawn; the outline is the area. The look is `00-demand`'s: magma,
white edges, a colorbar.


In [ ]:
import h3
from shapely.geometry import Polygon

capture = demand.attrs["capture_share"]
hexes = gpd.GeoDataFrame(demand[["demand"]],
                         geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in demand.index], crs=4326)
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
hexes.plot(column="demand", ax=ax, cmap="magma", linewidth=0.1, edgecolor="white", legend=True,
           legend_kwds={"label": "orders per day per hex", "shrink": 0.8})
gpd.GeoSeries([area], crs=4326).boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_aspect(1 / np.cos(np.radians(28.44)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Demand at default capture share {capture:.3f}, whole area ({len(hexes):,} hexes, H3 res 9)")
fig.savefig(FIGS / "03-demand.png", dpi=120)
plt.show()
print(f"hexes {len(demand):,}; total demand {demand.demand.sum():,.0f} orders/day; capture share {capture:.4f}; "
      f"per hex: median {demand.demand.median():.1f}, max {demand.demand.max():.1f}; res {demand.attrs['res']}")


![Demand per hex across the area](figures/03-demand.png)


Results from the run on 2026-10-29 (copied here because outputs are stripped on commit): **4,859 hexes at H3 res 9**,
total demand **89,728 orders a day** at the default capture share of **0.0805** (about 8% of households order on a
given day). Per hex the median is 11.2 orders a day and the maximum 110.3. These totals are at res 9; a res-8 build
would give slightly different totals.

The brightest hexes are in the north-east (the South Delhi end of the area); the middle band is moderate, and the
west and south edges are dark.


## Area demand map

Demand per res-9 hex across the whole area, in orders a day, at the default capture share (read from the demand
table's attributes). Every hex of `demand-r9` is drawn; the outline is the area. The look is `00-demand`'s: magma,
white edges, a colorbar.


In [ ]:
capture = demand.attrs["capture_share"]
hexes = gpd.GeoDataFrame(demand[["demand"]],
                         geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in demand.index], crs=4326)
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
hexes.plot(column="demand", ax=ax, cmap="magma", linewidth=0.1, edgecolor="white", legend=True,
           legend_kwds={"label": "orders per day per hex", "shrink": 0.8})
gpd.GeoSeries([area], crs=4326).boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_aspect(1 / np.cos(np.radians(28.44)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Demand at default capture share {capture:.3f}, whole area ({len(hexes):,} hexes, H3 res 9)")
fig.savefig(FIGS / "03-demand.png", dpi=120)
plt.show()
print(f"hexes {len(demand):,}; total demand {demand.demand.sum():,.0f} orders/day; capture share {capture:.4f}; "
      f"per hex: median {demand.demand.median():.1f}, max {demand.demand.max():.1f}; res {demand.attrs['res']}")


![Demand per hex across the area](figures/03-demand.png)


Results from the run on 2026-10-29 (copied here because outputs are stripped on commit): **4,859 hexes at H3 res 9**,
total demand **89,728 orders a day** at the default capture share of **0.0805** (about 8% of households order on a
given day). Per hex the median is 11.2 orders a day and the maximum 110.3. These totals are at res 9. The res-8 demand
table is built natively and is not comparable: it has 697 hexes and a total of 88,472 (1.4% below res 9's 89,728) orders a day, because it drops
the hexes whose centre is outside the area (see the "Demand and reach (stage 1) design" entry in the decision log).

The brightest hexes are in the north-east (the South Delhi end of the area); the middle band is moderate, and the
west and south edges are dark.


## Full-area solve time

The default scenario has to return a plan in 10 s or less. 0.5 extrapolated the solve from the slice (3.1-5.6 s for
N = 10 on the full area; three slice copies at the area's pair count solved in 1.2 s). Here the 0.5 CP-SAT model runs on
the real full-area tables: all of `demand-r9`'s hexes, the candidate sites in rank order, and the pairs of the table
reachable at the default scenario. The model is copied inline from `02-plan` (stage 2 moves it into the package),
changed only where the case is built: it reads the area's tables, not the slice's, and the constants come from
`assumptions.py`. The placeholder network is placed over the area's candidate sites by the same rule as on the slice.

**Main run: the placeholder network.** 3 operator, 4 Competitor A and 3 Competitor B stores (10, sized for the slice
and kept as it is: a deliberately thin network for an area that holds about 57 stores' worth of orders), N = 10 new
stores, default capacity. The time is `solve_plan` as a whole, model build plus solve. **Sensitivity run:** a network
sized to the area, round(area demand ÷ orders per store per day) stores, with each brand's share of `BRAND_COUNTS` (3:4:3) kept. More existing stores
means fewer open sites and, through competitor reach, different shares. Both run at X = 30 (1,458 candidate sites).

The plan rows of the assumptions table follow. The rows that say "slice" (brands and store counts, store placement, sites
for new stores) hold here for the area's candidate sites; the counts are the slice's placeholder for the main run.


In [ ]:
rows = assumptions.table()
plan_rows = ["Default scenario", "New stores, N", "Orders per store per day", "Capacity per store", "Brands and store counts",
             "Store placement", "Competitor reach", "Demand split", "Sites for new stores", "Assignment tie-break",
             "Integer scale", "Solver"]
pd.set_option("display.max_colwidth", None)
display(rows[rows.Input.isin(plan_rows)].style.hide(axis="index").set_properties(**{"text-align": "left"}))


In [ ]:
CAP_INT = round(CAPACITY * SCALE)

def reachable(ff_min):
    """Hex reachable from a site: free-flow minutes x multiplier + pick-and-pack <= promise."""
    return ff_min * MULTIPLIER + PICK_AND_PACK <= PROMISE + 1e-9

@dataclass
class Case:
    res: int
    hexes: pd.Index
    demand: np.ndarray        # orders per day per hex
    sites: pd.Index           # candidate sites, in rank order
    rank: np.ndarray          # places-count rank in the whole area, 0 = busiest
    ps: np.ndarray            # reachable pairs: site position
    ph: np.ndarray            # reachable pairs: hex position
    pm: np.ndarray            # reachable pairs: free-flow minutes
    n_area_hexes: int         # hexes in the whole area
    stores: dict              # brand -> site positions
    k: np.ndarray             # competitor brands reaching each hex
    share_int: np.ndarray     # operator share of each hex if it serves it, hundredths of an order
    new_sites: np.ndarray     # site positions open for a new store
    cap_int: int              # capacity, hundredths of an order

def load_case(res, hexes, candidates, table):
    """The full area at one resolution: every hex of the demand table, every candidate site in rank order, and the
    table's pairs reachable at the default scenario."""
    cand = candidates.sort_values("rank")
    tab = table[reachable(table.minutes)]
    spos = pd.Series(np.arange(len(cand)), index=cand.index)
    hpos = pd.Series(np.arange(len(hexes)), index=hexes.index)
    d = hexes["demand"].to_numpy()
    return Case(res, hexes.index, d, cand.index, cand["rank"].to_numpy(), spos[tab.candidate].to_numpy(),
                hpos[tab.hex].to_numpy(), tab.minutes.to_numpy(), len(hexes), {}, np.zeros(len(d), int),
                np.round(d * SCALE).astype(np.int64), np.arange(len(cand)), CAP_INT)

def describe(case):
    per_hex = np.bincount(case.ph, minlength=len(case.hexes))
    print(f"res {case.res}: hexes {len(case.hexes):,}; candidate sites {len(case.sites):,}; default-scenario pairs {len(case.ps):,}")
    print(f"hexes no candidate reaches: {(per_hex == 0).sum():,}, demand {case.demand[per_hex == 0].sum():,.0f} of {case.demand.sum():,.0f} orders/day")
    print(f"largest catchment's demand {np.bincount(case.ps, weights=case.demand[case.ph]).max():,.0f} orders/day against capacity {CAPACITY:,.0f}")

area_9 = load_case(9, demand, cand_cmd, table_cmd)
describe(area_9)
print(f"table: {len(table_cmd):,} pairs within {assumptions.MAX_PAIR_MIN} min")


In [ ]:
def place_network(C, counts=BRAND_COUNTS, seed=SEED):
    rng = np.random.default_rng(seed)
    reached = np.bincount(C.ps, weights=C.demand[C.ph], minlength=len(C.sites))
    chosen = rng.choice(len(C.sites), size=sum(counts.values()), replace=False, p=reached / reached.sum())
    labels = rng.permutation(np.repeat(list(counts), list(counts.values())))
    return {b: sorted(chosen[labels == b].tolist()) for b in counts}

def with_network(C, stores, cap_int=None):
    comp = [b for b in stores if b != "Operator"]
    hit = np.zeros((len(comp), len(C.hexes)), bool)
    for i, b in enumerate(comp):
        hit[i, C.ph[np.isin(C.ps, stores[b])]] = True
    k = hit.sum(axis=0)
    occupied = np.array(sorted(s for ss in stores.values() for s in ss))
    assert len(occupied) == len(set(occupied)), "two stores on one site"
    return replace(C, stores=stores, k=k, share_int=np.round(C.demand / (1 + k) * SCALE).astype(np.int64),
                   new_sites=np.setdiff1d(np.arange(len(C.sites)), occupied), cap_int=cap_int or C.cap_int)

def solve_plan(C, N, new_sites=None, pair_model=False, time_limit=TIME_LIMIT):
    """Choose N new sites from new_sites (default: all open ones) to maximise the operator's orders.
    pair_model=True builds the plain model (x for every pair); otherwise only sites with W_s > capacity get x."""
    t0 = time.perf_counter()
    new = C.new_sites if new_sites is None else new_sites
    op = np.array(C.stores["Operator"])
    is_new, is_op = np.zeros(len(C.sites), bool), np.zeros(len(C.sites), bool)
    is_new[new], is_op[op] = True, True
    keep = (is_new | is_op)[C.ps]
    ps, ph = C.ps[keep], C.ph[keep]
    reach_w = np.bincount(ps, weights=C.share_int[ph], minlength=len(C.sites))      # W_s
    constrained = np.ones(len(C.sites), bool) if pair_model else reach_w > C.cap_int
    model = cp_model.CpModel()
    y = {int(s): model.NewBoolVar(f"y{s}") for s in new}
    model.Add(sum(y.values()) == N)
    terms, coeffs, const = [], [], 0
    x_at = defaultdict(list)
    for h, sites_h in pd.Series(ps).groupby(ph):
        sites_h = sites_h.to_numpy()
        w = int(C.share_int[h])
        free = sites_h[~constrained[sites_h]]
        if is_op[free].any():               # an existing store that cannot hit capacity covers this hex
            const += w
            continue
        row = []
        if len(free):
            covered = model.NewBoolVar("")
            model.AddBoolOr([y[int(s)] for s in free]).OnlyEnforceIf(covered)
            row.append(covered)
        for s in sites_h[constrained[sites_h]]:
            xv = model.NewBoolVar("")
            if is_new[s]:
                model.AddImplication(xv, y[int(s)])
            x_at[int(s)].append((xv, w))
            row.append(xv)
        if len(row) > 1:
            model.AddAtMostOne(row)
        terms += row
        coeffs += [w] * len(row)
    for lst in x_at.values():
        model.Add(cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst]) <= C.cap_int)
    model.Maximize(cp_model.LinearExpr.WeightedSum(terms, coeffs) + const)
    t1 = time.perf_counter()
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_workers = WORKERS
    solver.parameters.random_seed = SOLVER_SEED
    status = solver.Solve(model)
    t2 = time.perf_counter()
    ok = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    return {"status": solver.StatusName(status), "chosen": sorted(s for s in y if solver.Value(y[s])) if ok else [],
            "obj": solver.ObjectiveValue() / SCALE, "bound": solver.BestObjectiveBound() / SCALE,
            "pairs": len(ps), "new_sites": len(new), "hexes": int(len(np.unique(ph))),
            "constrained sites": int(constrained[np.unique(ps)].sum()),
            "build_s": t1 - t0, "solve_s": t2 - t1, "total_s": t2 - t0}

def evaluate(C, open_sites):
    """Best assignment of hexes to the operator's open stores within capacity. Returns (owner per hex, orders x 100)."""
    keep = np.isin(C.ps, list(open_sites))
    ps, ph, pm = C.ps[keep], C.ph[keep], C.pm[keep]
    w = C.share_int[ph]
    model = cp_model.CpModel()
    x = [model.NewBoolVar("") for _ in range(len(ps))]
    for _, js in pd.Series(np.arange(len(ps))).groupby(ph):
        if len(js) > 1:
            model.AddAtMostOne([x[j] for j in js])
    for _, js in pd.Series(np.arange(len(ps))).groupby(ps):
        js = js.to_numpy()
        model.Add(cp_model.LinearExpr.WeightedSum([x[j] for j in js], [int(w[j]) for j in js]) <= C.cap_int)
    served = cp_model.LinearExpr.WeightedSum(x, [int(v) for v in w])
    model.Maximize(served)                      # stage 1: orders served; always solved to optimality
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = SOLVER_SEED
    solver.parameters.max_time_in_seconds = 30
    status = solver.Solve(model)
    assert status == cp_model.OPTIMAL, solver.StatusName(status)
    best = round(solver.ObjectiveValue())
    for v in x:
        model.AddHint(v, solver.Value(v))
    model.Add(served == best)                   # stage 2: the same orders, the least total travel time (hundredths of a minute)
    model.Minimize(cp_model.LinearExpr.WeightedSum(x, [round(float(m) * 100) for m in pm]))
    solver.parameters.max_time_in_seconds = 5   # best effort: stage 1 fixes the orders, this only picks among ties
    status = solver.Solve(model)
    assert status in (cp_model.OPTIMAL, cp_model.FEASIBLE), solver.StatusName(status)
    owner = np.full(len(C.hexes), -1)
    for j in np.flatnonzero([solver.Value(v) for v in x]):
        owner[ph[j]] = ps[j]
    assert int(C.share_int[owner >= 0].sum()) == best
    return owner, best


### The runs

Each run builds the case (`place_network`, `with_network`), then times `solve_plan` three times, build and solve
together. After the timed runs the CP-SAT objective is checked against `evaluate` for the chosen stores, as in 02-plan.
The 60 s time limit and 8 workers are the assumptions table's.


In [ ]:
def timed_runs(C, N, new_sites=None, repeats=3):
    runs = []
    for _ in range(repeats):
        load = os.getloadavg()[0]
        r = solve_plan(C, N, new_sites=new_sites)
        r["load_before"] = load
        runs.append(r)
    return runs

def summary(name, C, runs, check="optimal"):
    """check: 'always' runs evaluate even if CP-SAT is not proven optimal (to time it); 'optimal' only if proven; 'never'."""
    r = runs[0]
    t = np.array([x["total_s"] for x in runs])
    for i, x in enumerate(runs):
        print(f"{name}, run {i + 1}: {x['status']}, objective {x['obj']:.2f}, bound {x['bound']:.2f}, build {x['build_s']:.2f} s, "
              f"solve {x['solve_s']:.2f} s, total {x['total_s']:.2f} s, load before {x['load_before']:.1f}")
    same = all(x["status"] == r["status"] and abs(x["obj"] - r["obj"]) < 0.005 for x in runs)   # alternative optima may pick different sites
    # the CP-SAT objective must equal what the evaluator serves for the chosen stores
    ev, ev_s = float("nan"), float("nan")
    if check == "always" or (check == "optimal" and r["status"] == "OPTIMAL"):
        ev_s = time.perf_counter()
        try:
            _, served_int = evaluate(C, list(C.stores["Operator"]) + r["chosen"])
            ev = served_int / SCALE
            if r["status"] == "OPTIMAL":
                assert abs(r["obj"] - ev) < 0.005, (r["status"], r["obj"], ev)
        except AssertionError as e:
            if r["status"] == "OPTIMAL" and "FEASIBLE" not in str(e):
                raise
            # evaluate's own stage 1 did not reach OPTIMAL within its 30 s limit
        ev_s = time.perf_counter() - ev_s
    print(f"{name}: evaluate serves {ev:.2f} for run 1's stores (CP-SAT objective {r['obj']:.2f}), {ev_s:.1f} s; runs agree: {same}")
    catch = np.bincount(C.ps, weights=C.demand[C.ph], minlength=len(C.sites))
    return {"network": name, "existing stores (op/A/B)": "/".join(str(len(C.stores[b])) for b in C.stores),
            "status": r["status"], "objective (orders/day)": round(r["obj"], 2), "bound": round(r["bound"], 2),
            "pairs": r["pairs"], "hexes": r["hexes"], "new sites": r["new_sites"], "constrained sites": r["constrained sites"],
            "largest catchment (orders/day)": round(catch.max()), "capacity": round(CAPACITY),
            "build s (median)": round(float(np.median([x["build_s"] for x in runs])), 2),
            "solve s (median)": round(float(np.median([x["solve_s"] for x in runs])), 2),
            "total s median": round(float(np.median(t)), 2), "min": round(float(t.min()), 2), "max": round(float(t.max()), 2),
            "all totals": ", ".join(f"{v:.2f}" for v in t), "evaluate s": round(ev_s, 1), "evaluate orders/day": round(ev, 2)}

print(f"OR-tools {ortools.__version__}; os.cpu_count() {os.cpu_count()}; load average at start (1, 5, 15 min): "
      + ", ".join(f"{v:.2f}" for v in os.getloadavg()))

# main run: the placeholder network (3/4/3), N = 10
placeholder_9 = with_network(area_9, place_network(area_9))
runs_main = timed_runs(placeholder_9, N_AREA)
res_main = summary("X=30 placeholder 3/4/3", placeholder_9, runs_main, check="always")

# sensitivity run: a network sized to the area, split 3:4:3 in proportion
n_net = round(area_9.demand.sum() / ORDERS_PER_STORE_DAY)
counts_sized = {b: round(n_net * n / sum(BRAND_COUNTS.values())) for b, n in BRAND_COUNTS.items()}   # BRAND_COUNTS' proportions
assert sum(counts_sized.values()) == n_net
sized_9 = with_network(area_9, place_network(area_9, counts_sized))
runs_sized = timed_runs(sized_9, N_AREA)
res_sized = summary(f"X=30 sized to the area {'/'.join(map(str, counts_sized.values()))}", sized_9, runs_sized)

display(pd.DataFrame([res_main, res_sized]).set_index("network").T)
print("load average (1 min) before each timed run, main:", [round(x["load_before"], 2) for x in runs_main],
      "sensitivity:", [round(x["load_before"], 2) for x in runs_sized])
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))
print("area demand", f"{area_9.demand.sum():,.0f}", "÷", f"{ORDERS_PER_STORE_DAY:,.1f}", "=", f"{area_9.demand.sum() / ORDERS_PER_STORE_DAY:.2f}", "->", n_net, counts_sized)


Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not these). Apple M1, 8 cores, 8 GB RAM, OR-tools 9.15.6755, 8 workers, 60 s limit, solver seed 1. **The machine
was loaded by other jobs: the 1-minute load average was 13 to 28 before the main runs (the first at 13, then 26 and 28),
22 to 24 before the sensitivity runs and 22 to 30 before the cuts and the res-8 runs, on 8 cores.** The times below are
inflated by that, and nothing here was measured on an idle machine. It cannot change the main verdict, since those
runs stopped at the 60 s limit, but the OPTIMAL runs would be faster on a quiet machine: the area-sized network took 14 s
here, 21 s in an earlier run of this cell and 30 to 42 s in the first, at higher loads. Two runs of the main network also
found different plans (objective 28,234 here, 27,965 earlier), as an unproven 8-worker search does.

The case: 4,859 hexes, 1,458 candidate sites, 59,855 default-scenario pairs (free-flow ≤ 5.0 min; 1,187,469 in the
table). 1,557 hexes with 5,839 of the area's 89,728 orders a day are reached by no candidate (6.5%), so coverage at
X = 30 is 93.5%. The largest catchment holds **5,266 orders a day against a capacity of 2,364**.

| X = 30, N = 10 | main: placeholder 3/4/3 | sensitivity: sized to the area, 17/23/17 (57 stores) |
|---|---:|---:|
| status | FEASIBLE (stopped at the 60 s limit) | OPTIMAL |
| total time, median (min / max) | **60.9 s** (60.6 / 61.8) | **14.3 s** (12.4 / 15.4) |
| build / solve, median | 0.6 s / 60.2 s | 0.2 s / 14.1 s |
| objective, orders a day | 28,234 (runs: 28,234, 28,234, 28,266) | 30,401.05 |
| bound | 33,430 (18.4% above the objective) | 30,401.05 |
| pairs, hexes, sites open for new stores | 59,532, 3,302, 1,448 | 57,944, 3,300, 1,401 |
| constrained sites (W_s > capacity) | **123** | 22 |
| `evaluate` for the chosen stores | not finished: its first solve hit its 30 s limit without a proof | 30,401.05, equal to the objective (6.0 s) |

Area demand ÷ orders per store per day = 89,728 ÷ 1,576.1 = 56.9, so the sensitivity network has 57 stores: each brand's
share of `BRAND_COUNTS` (3:4:3) of 57, rounded, is 17 operator, 23 Competitor A, 17 Competitor B. The same seed places
both networks.

**The main run misses the 10 s bar by a factor of six or more**, and it is not close to proven: after 60 s the best plan is
still 18.4% below the bound. The sensitivity network misses too (14 s, though it is proven optimal). **The cause is
capacity binding on the area.** 0.5 measured the slice, where the largest catchment (1,981 orders a day) was under the
capacity (2,364), no site was constrained and the problem was maximum coverage, extrapolated to 3.1-5.6 s (three slice
copies: 1.2 s). On the area the dense north-east gives catchments of up to 5,266 orders a day, 123 sites exceed
capacity, and each needs assignment variables and a knapsack constraint. The diagnostic below confirms it: the same
network and sites with capacity effectively infinite solve to proven optimality in 2.7 s. This is the 0.5 open point
"capacity that binds is slow", which turns out to bite at the default scenario. For the same reason `evaluate` (the
scorer, which assigns hexes to a plan's stores from scratch) is slow when capacity binds: its first solve did not reach
OPTIMAL in 30 s for the main run's 13 stores. That check was skipped for the X cuts and res 8 (it is only made when
CP-SAT proves optimality, and none did).


### Cutting X

If the default scenario misses 10 s, the 0.5 entry says to cut X before the resolution. X is the share of the area's hexes
that may hold a new store. The X% candidate set is the X = 30 set with `rank < n_top(X, n_hexes, n_with_places)`: a
hex's rank does not depend on X, the candidates are those under that cut-off, and a pair's minutes depend only on its two
endpoints' snaps, so the X% table is the X = 30 table filtered to those candidates. This is equivalent to
`catchment-build reach --x-pct X` and needs no OSRM run. `n_top` is the package's.

The network is placed once, at X = 30, and keeps its stores at every X: only the sites open to new stores shrink,
`new = new_sites[rank[new_sites] < k]` as in the 02-plan grid. X goes down 20, 10, 5 and stops at the first X whose median
total is at most 10 s with status OPTIMAL. Coverage is the share of the area's demand on hexes that some candidate site
at that X reaches at the default scenario. The last row is a labelled diagnostic.


In [ ]:
n_hexes, n_with_places = len(demand), len(cand_cmd)   # all of demand-r9 lies in the area; every X=30 candidate has a place
assert n_top(30, n_hexes, n_with_places) == len(cand_cmd)   # and 1,458 exceeds n_top at every smaller X, so the min never binds

def coverage(case, cutoff):
    """Share of the area's demand on hexes that a candidate site with rank < cutoff reaches at the default scenario."""
    hit = np.unique(case.ph[case.rank[case.ps] < cutoff])
    return case.demand[hit].sum() / case.demand.sum()

cov30 = coverage(area_9, len(area_9.sites))
res_x = [dict(res_main, X=30, candidates=len(area_9.sites), coverage=cov30)]
fits = None
for x_pct in (20, 10, 5):
    cutoff = n_top(x_pct, n_hexes, n_with_places)
    new = placeholder_9.new_sites[placeholder_9.rank[placeholder_9.new_sites] < cutoff]
    runs = timed_runs(placeholder_9, N_AREA, new_sites=new)
    row = dict(summary(f"X={x_pct}", placeholder_9, runs), X=x_pct, candidates=cutoff, coverage=coverage(area_9, cutoff))
    res_x.append(row)
    if row["status"] == "OPTIMAL" and row["total s median"] <= SOLVE_BAR_S:
        fits = x_pct
        break
print("X that fits:", fits)

# diagnostic, not a scenario: the same network and X = 30 with capacity effectively infinite (plain maximum coverage)
infinite_9 = with_network(area_9, placeholder_9.stores, cap_int=10**12)
res_inf = dict(summary("X=30 DIAGNOSTIC infinite capacity", infinite_9, timed_runs(infinite_9, N_AREA), check="never"),
               X="30, infinite capacity", candidates=len(area_9.sites), coverage=cov30)
cols = ["X", "candidates", "coverage", "status", "objective (orders/day)", "bound", "constrained sites", "new sites", "pairs",
        "build s (median)", "solve s (median)", "total s median", "min", "max", "all totals"]
display(pd.DataFrame(res_x + [res_inf])[cols].set_index("X").round(4))
print(f"coverage at X=30: {cov30:.4f}; " + ", ".join(f"X={r['X']}: {r['coverage']:.4f}" for r in res_x[1:]))
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))


**Cutting X, then the diagnostic.** All rows are the main network (3/4/3, seed 2026, placed once at X = 30), N = 10,
default capacity, median of three runs. Coverage is the share of the area's demand reached by some candidate site at
the default scenario. The objective is run 1's.

| X | candidate sites | coverage | status | total time, median (min / max) | objective | bound | constrained sites |
|---|---:|---:|---|---:|---:|---:|---:|
| 30 | 1,458 | 93.5% | FEASIBLE | 60.9 s (60.6 / 61.8) | 28,234 | 33,430 | 123 |
| 20 | 972 | 88.3% | FEASIBLE | 60.8 s (60.7 / 61.0) | 28,267 | 33,134 | 91 |
| 10 | 486 | 79.8% | FEASIBLE | 60.4 s (60.3 / 60.8) | 27,965 | 32,325 | 53 |
| 5 | 243 | 71.2% | FEASIBLE | 60.4 s (60.4 / 60.5) | 27,356 | 30,578 | 30 |
| **diagnostic**: X = 30, capacity effectively infinite | 1,458 | 93.5% | OPTIMAL | 2.7 s (2.6 / 3.0) | 33,697 | 33,697 | 0 |

**No X fits.** Every cut runs to the 60 s limit with an unproven plan, down to X = 5 (243 sites, 30 constrained sites, a
bound still 11.8% above the best plan), at a coverage cost of 5 points at X = 20, 14 at X = 10 and 22 at X = 5. Cutting
sites shrinks the model but leaves the constrained sites that make it hard. The X% candidate sets here are the X = 30
table filtered by rank, equivalent to `catchment-build reach --x-pct X` (a hex's rank does not depend on X and a pair's
minutes only on its two endpoints' snaps). The existing stores stay where they were placed at X = 30 at every X; only
new-store sites shrink.


### Res 8, the last resort

No X fits, so the last resort is the coarser grid. `catchment-build reach --res 8` (X = 30) gives 697 hexes and 210
candidate sites; the table here was written by `build_reach` and `build.py`'s `write_parquet` against a separate OSRM
container. The network is placed with the same rule (3/4/3, seed 2026, probability proportional to the demand a site
reaches at the default scenario) over the res-8 candidates; N = 10 and the capacity are unchanged.

**Caveat: res-8 and res-9 numbers are not comparable.** The res-8 demand table is built natively at res 8, which drops the
population of res-8 hexes whose centre is outside the area (941,794 households against 955,163 at res 9, about 1.4%
lower; see the "Demand and reach (stage 1) design" entry in the decision log). Its totals, coverage and objective rest on
a slightly smaller demand.


In [ ]:
demand_r8 = pd.read_parquet(DATA / "demand-r8.parquet")
cand_r8 = pd.read_parquet(DATA / "candidates-r8-x30.parquet")
table_r8 = pd.read_parquet(DATA / "table-r8-x30.parquet")
print(f"households: res 8 {demand_r8.households.sum():,.0f} against res 9 {demand.households.sum():,.0f} "
      f"({demand_r8.households.sum() / demand.households.sum() - 1:+.2%}); demand: res 8 {demand_r8.demand.sum():,.0f} against "
      f"res 9 {demand.demand.sum():,.0f} orders/day ({demand_r8.demand.sum() / demand.demand.sum() - 1:+.2%})")
area_8 = load_case(8, demand_r8, cand_r8, table_r8)
describe(area_8)
print(f"table: {len(table_r8):,} pairs within {assumptions.MAX_PAIR_MIN} min")
placeholder_8 = with_network(area_8, place_network(area_8))
runs_8 = timed_runs(placeholder_8, N_AREA)
res_8 = dict(summary("res 8 X=30 placeholder 3/4/3", placeholder_8, runs_8), X=30, candidates=len(area_8.sites),
             coverage=coverage(area_8, len(area_8.sites)))
display(pd.DataFrame([dict(res_x[0], res=9), dict(res_8, res=8)])[["res"] + cols].set_index("res").T)
print(f"res 8 coverage {res_8['coverage']:.4f} against res 9 {cov30:.4f}; verdict against {SOLVE_BAR_S} s: "
      f"{res_8['status']}, median {res_8['total s median']} s")
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))


Results from the same run (the table: `candidates-r8-x30` 210 sites, `table-r8-x30` 24,700 pairs; the container was
`catchment-osrm-15` on port 5015, 29 s including the table build). **Res-8 numbers are not comparable with res 9's** (the
caveat above): its demand is 88,472 orders a day on 697 hexes, 1.4% below res 9's 89,728.

| X = 30, main network 3/4/3 | res 9 | res 8 |
|---|---:|---:|
| hexes, candidate sites | 4,859, 1,458 | 697, 210 |
| coverage of demand at the default scenario | 93.5% | 78.9% |
| default-scenario pairs, kept in the run | 59,855, 59,532 | 1,194, 1,140 |
| constrained sites (W_s > capacity) | 123 | 18 |
| largest catchment's demand vs capacity | 5,266 vs 2,364 | 4,852 vs 2,364 |
| status | FEASIBLE | FEASIBLE |
| total time, median (min / max) | 60.9 s (60.6 / 61.8) | **60.3 s** (60.1 / 60.3) |
| objective, bound | 28,234; 33,430 | 26,619 (in all three runs); 29,309 (10.1% above) |

**Res 8 misses too.** It is the coarsest grid on the table, 1,140 pairs, and still unproven after 60 s, at a coverage 14.6
points lower. Size is not what makes the problem slow: capacity binds at 18 sites here too (the 0.5 entry's 0.04 s for
res 8 assumed none), and the capacity-bound assignment is what CP-SAT cannot close. **So the bar is missed at every
setting the ticket allows (X = 30, 20, 10, 5 at res 9, and res 8 at X = 30), and res 9 and X = 30 stand.** Not tried:
res 8 with X cut as well; it would shrink a model that is already not size-bound.

**Stage 2 needs a decision** on how to solve the default scenario in time: a faster model for binding capacity, a warm
start from the greedy plan, or the capacity default (2,364 is 1.5 times the average, and the area's dense catchments are
more than twice that). The infinite-capacity row shows the coverage part alone is fast (2.7 s here).


## Next

The spot-check follows in another issue of the stage 1 PRD (#9).
